**IMPORTING NECESSARY LIBRARIES**

In [193]:
import os
import numpy as np
import pandas as pd 
import seaborn as sns
import requests
from bs4 import BeautifulSoup as soup
from newspaper import Article
import nltk
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import classification_report, accuracy_score , confusion_matrix
import seaborn as sns
from textblob import TextBlob
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from newspaper import Article, ArticleException
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.preprocessing import FunctionTransformer
from nltk.sentiment.vader import SentimentIntensityAnalyzer
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.metrics import precision_recall_fscore_support
from sklearn.svm import SVC
import warnings
warnings.filterwarnings("ignore",category=DeprecationWarning)

In [194]:
import sys
import os

module_path = os.path.abspath(os.path.join('..', 'utils')) 

if module_path not in sys.path:
    sys.path.append(module_path)

from custom_utils import VaderSentimentExtractor

In [195]:
nltk.download('vader_lexicon')

[nltk_data] Downloading package vader_lexicon to
[nltk_data]     C:\Users\vedan\AppData\Roaming\nltk_data...
[nltk_data]   Package vader_lexicon is already up-to-date!


True

**LOADING DATASET**

In [196]:
root = r"C:\Users\vedan\Downloads/data/"
art_list = os.listdir(root)
print(f"total articles {len(art_list)}")
print("sample titles: ")
print(*art_list[:3], sep='\n')

total articles 7204
sample titles: 
20181010_business_india-business_bob-partners-truecaller-for-upi-payments.txt
20181010_business_india-business_city-to-get-a-centre-of-excellence-for-fintech.txt
20181010_business_india-business_cos-should-offer-flexible-work-policies-to-attract-best-talent.txt


**DATA CLEANING**

In [197]:
contents = []
for filename in art_list:

    file_path = os.path.join(root, filename)
    
  
    with open(file_path, 'r', encoding='utf-8') as f:
        
        contents.append(f.read())

In [198]:
df = pd.DataFrame(list(zip(art_list, contents)), columns=['title', 'content'])
df.head()

,title,content
0,20181010_business_india-business_bob-partners-...,[Mumbai: Bank of Baroda (BoB) has tied-up with...
1,20181010_business_india-business_city-to-get-a...,[Chennai will soon be home to a Centre of Exce...
2,20181010_business_india-business_cos-should-of...,"[By Kamal Karanth According to Aristotle, ‘The..."
3,20181010_business_india-business_deal-flows-cu...,[CHENNAI: The traditionally strong quarter (Ju...
4,20181010_business_india-business_equity-mutual...,[Coimbatore: FIIs (foreign institutional inves...


In [199]:
df['date'] = df.title.apply(lambda x: int(x.split('_')[0]))
df['tag'] = df.title.apply(lambda x: ("_".join(x.split('-')[0].split('_')[1:-1])))
df.head()

,title,content,date,tag
0,20181010_business_india-business_bob-partners-...,[Mumbai: Bank of Baroda (BoB) has tied-up with...,20181010,business
1,20181010_business_india-business_city-to-get-a...,[Chennai will soon be home to a Centre of Exce...,20181010,business
2,20181010_business_india-business_cos-should-of...,"[By Kamal Karanth According to Aristotle, ‘The...",20181010,business
3,20181010_business_india-business_deal-flows-cu...,[CHENNAI: The traditionally strong quarter (Ju...,20181010,business
4,20181010_business_india-business_equity-mutual...,[Coimbatore: FIIs (foreign institutional inves...,20181010,business


In [200]:
months = [i for i in range(1, 13)]
days = [i for i in range(1, 32)]

def convert(x):
    x = str(x)
    splits = [(int(x[:k]), int(x[k:])) for k in range(1, len(x))]
    for i, j in splits:
        if i in months and j in days: 
            return i, j

df['year'] = df.title.apply(lambda x: int(x.split('_')[0][:4]))
rest = df.title.apply(lambda x: int(x.split('_')[0][4:]))
a = rest.apply(convert)
df['month'] = [i[0] for i in a]
df['day'] = [i[1] for i in a]
df.head()

,title,content,date,tag,year,month,day
0,20181010_business_india-business_bob-partners-...,[Mumbai: Bank of Baroda (BoB) has tied-up with...,20181010,business,2018,1,10
1,20181010_business_india-business_city-to-get-a...,[Chennai will soon be home to a Centre of Exce...,20181010,business,2018,1,10
2,20181010_business_india-business_cos-should-of...,"[By Kamal Karanth According to Aristotle, ‘The...",20181010,business,2018,1,10
3,20181010_business_india-business_deal-flows-cu...,[CHENNAI: The traditionally strong quarter (Ju...,20181010,business,2018,1,10
4,20181010_business_india-business_equity-mutual...,[Coimbatore: FIIs (foreign institutional inves...,20181010,business,2018,1,10


In [201]:
df['headline'] = df.title.apply(lambda x: (x.split('-')[0].split('_')[-1] + '-' + '-'.join(x.split('-')[1:])).replace("-", " ")[:-3])
df['content'] = df.content.apply(lambda x: x[1:-1])
df.head()

,title,content,date,tag,year,month,day,headline
0,20181010_business_india-business_bob-partners-...,Mumbai: Bank of Baroda (BoB) has tied-up with ...,20181010,business,2018,1,10,india business_bob partners truecaller for upi...
1,20181010_business_india-business_city-to-get-a...,Chennai will soon be home to a Centre of Excel...,20181010,business,2018,1,10,india business_city to get a centre of excelle...
2,20181010_business_india-business_cos-should-of...,"By Kamal Karanth According to Aristotle, ‘The ...",20181010,business,2018,1,10,india business_cos should offer flexible work ...
3,20181010_business_india-business_deal-flows-cu...,CHENNAI: The traditionally strong quarter (Jul...,20181010,business,2018,1,10,india business_deal flows currency move to hel...
4,20181010_business_india-business_equity-mutual...,Coimbatore: FIIs (foreign institutional invest...,20181010,business,2018,1,10,india business_equity mutual funds remain bull...


In [202]:
def get_loc(x):
    p = x.split(':')[0]
    if len(p.split(" ")) < 6:
        return p
    elif len(p.split('()')[0]) < 30:
        return p.split(',')[0]
    return ""

df['loc'] = df['content'].apply(get_loc)
df.head()

,title,content,date,tag,year,month,day,headline,loc
0,20181010_business_india-business_bob-partners-...,Mumbai: Bank of Baroda (BoB) has tied-up with ...,20181010,business,2018,1,10,india business_bob partners truecaller for upi...,Mumbai
1,20181010_business_india-business_city-to-get-a...,Chennai will soon be home to a Centre of Excel...,20181010,business,2018,1,10,india business_city to get a centre of excelle...,
2,20181010_business_india-business_cos-should-of...,"By Kamal Karanth According to Aristotle, ‘The ...",20181010,business,2018,1,10,india business_cos should offer flexible work ...,
3,20181010_business_india-business_deal-flows-cu...,CHENNAI: The traditionally strong quarter (Jul...,20181010,business,2018,1,10,india business_deal flows currency move to hel...,CHENNAI
4,20181010_business_india-business_equity-mutual...,Coimbatore: FIIs (foreign institutional invest...,20181010,business,2018,1,10,india business_equity mutual funds remain bull...,Coimbatore


In [203]:
df = df[['date','year','month' ,'day', 'tag', 'loc', 'headline', 'title', 'content']]
df.head()

,date,year,month,day,tag,loc,headline,title,content
0,20181010,2018,1,10,business,Mumbai,india business_bob partners truecaller for upi...,20181010_business_india-business_bob-partners-...,Mumbai: Bank of Baroda (BoB) has tied-up with ...
1,20181010,2018,1,10,business,,india business_city to get a centre of excelle...,20181010_business_india-business_city-to-get-a...,Chennai will soon be home to a Centre of Excel...
2,20181010,2018,1,10,business,,india business_cos should offer flexible work ...,20181010_business_india-business_cos-should-of...,"By Kamal Karanth According to Aristotle, ‘The ..."
3,20181010,2018,1,10,business,CHENNAI,india business_deal flows currency move to hel...,20181010_business_india-business_deal-flows-cu...,CHENNAI: The traditionally strong quarter (Jul...
4,20181010,2018,1,10,business,Coimbatore,india business_equity mutual funds remain bull...,20181010_business_india-business_equity-mutual...,Coimbatore: FIIs (foreign institutional invest...


**SENTIMENT ANALYSIS USING TEXTBLOB**

In [204]:
def get_sentiment(text):
    if text == '':
        return 0.0, 0.0
    analysis = TextBlob(text)
    return analysis.sentiment.polarity, analysis.sentiment.subjectivity

In [205]:
df[['polarity', 'subjectivity']] = df['content'].apply(
    lambda x: pd.Series(get_sentiment(x))
)

In [206]:
df

,date,year,month,day,tag,loc,headline,title,content,polarity,subjectivity
0,20181010,2018,1,10,business,Mumbai,india business_bob partners truecaller for upi...,20181010_business_india-business_bob-partners-...,Mumbai: Bank of Baroda (BoB) has tied-up with ...,0.293290,0.451491
1,20181010,2018,1,10,business,,india business_city to get a centre of excelle...,20181010_business_india-business_city-to-get-a...,Chennai will soon be home to a Centre of Excel...,-0.026154,0.355000
2,20181010,2018,1,10,business,,india business_cos should offer flexible work ...,20181010_business_india-business_cos-should-of...,"By Kamal Karanth According to Aristotle, ‘The ...",0.213180,0.475056
3,20181010,2018,1,10,business,CHENNAI,india business_deal flows currency move to hel...,20181010_business_india-business_deal-flows-cu...,CHENNAI: The traditionally strong quarter (Jul...,0.102381,0.409524
4,20181010,2018,1,10,business,Coimbatore,india business_equity mutual funds remain bull...,20181010_business_india-business_equity-mutual...,Coimbatore: FIIs (foreign institutional invest...,0.133001,0.397082
...,...,...,...,...,...,...,...,...,...,...,...
7199,20181027,2018,1,27,city_pune,PUNE,alert residents nab duo for robbing carpenter.,20181027_city_pune_alert-residents-nab-duo-for...,PUNE: Alert residents nabbed a man and his min...,0.021726,0.346726
7200,20181027,2018,1,27,city_pune,PUNE,as festival nears prices of bajra wheat jowar ...,20181027_city_pune_as-festival-nears-prices-of...,"PUNE: After onions, it’s now the turn of wheat...",0.024463,0.341648
7201,20181027,2018,1,27,city_pune,PUNE,eco friendly twist to this years lanterns for ...,20181027_city_pune_eco-friendly-twist-to-this-...,PUNE: Shops lined in the Raviwar Peth market a...,0.153783,0.447407
7202,20181027,2018,1,27,city_pune,PUNE,ferreira gonsalves back in police custody afte...,20181027_city_pune_ferreira-gonsalves-back-in-...,PUNE: The city police on Friday evening took a...,-0.005102,0.302551


**FORMING LABLES CONSIDERING SUBJECTIVITY PARAMETER**

In [207]:
SUBJECTIVITY_THRESHOLD = 0.5
df['bias_label'] = (df['subjectivity'] > SUBJECTIVITY_THRESHOLD).astype(int)

In [208]:
df

,date,year,month,day,tag,loc,headline,title,content,polarity,subjectivity,bias_label
0,20181010,2018,1,10,business,Mumbai,india business_bob partners truecaller for upi...,20181010_business_india-business_bob-partners-...,Mumbai: Bank of Baroda (BoB) has tied-up with ...,0.293290,0.451491,0
1,20181010,2018,1,10,business,,india business_city to get a centre of excelle...,20181010_business_india-business_city-to-get-a...,Chennai will soon be home to a Centre of Excel...,-0.026154,0.355000,0
2,20181010,2018,1,10,business,,india business_cos should offer flexible work ...,20181010_business_india-business_cos-should-of...,"By Kamal Karanth According to Aristotle, ‘The ...",0.213180,0.475056,0
3,20181010,2018,1,10,business,CHENNAI,india business_deal flows currency move to hel...,20181010_business_india-business_deal-flows-cu...,CHENNAI: The traditionally strong quarter (Jul...,0.102381,0.409524,0
4,20181010,2018,1,10,business,Coimbatore,india business_equity mutual funds remain bull...,20181010_business_india-business_equity-mutual...,Coimbatore: FIIs (foreign institutional invest...,0.133001,0.397082,0
...,...,...,...,...,...,...,...,...,...,...,...,...
7199,20181027,2018,1,27,city_pune,PUNE,alert residents nab duo for robbing carpenter.,20181027_city_pune_alert-residents-nab-duo-for...,PUNE: Alert residents nabbed a man and his min...,0.021726,0.346726,0
7200,20181027,2018,1,27,city_pune,PUNE,as festival nears prices of bajra wheat jowar ...,20181027_city_pune_as-festival-nears-prices-of...,"PUNE: After onions, it’s now the turn of wheat...",0.024463,0.341648,0
7201,20181027,2018,1,27,city_pune,PUNE,eco friendly twist to this years lanterns for ...,20181027_city_pune_eco-friendly-twist-to-this-...,PUNE: Shops lined in the Raviwar Peth market a...,0.153783,0.447407,0
7202,20181027,2018,1,27,city_pune,PUNE,ferreira gonsalves back in police custody afte...,20181027_city_pune_ferreira-gonsalves-back-in-...,PUNE: The city police on Friday evening took a...,-0.005102,0.302551,0


**AFTER TRAINING THE MODEL WE FOUND THAT TEXTBLOB DOES NOT RECOGNISE EMOTIONS AND OTHER IMPORTANT FACTORS NECESSARY FOR ACCURACY OF THE MODEL**

**THEREFORE WE USED VADER LIBRARY FOR SENTIMENT ANALYSIS**

In [209]:
analyzer = SentimentIntensityAnalyzer()

VADER_THRESHOLD = 0.5 
def relabel_with_vader(content):
    if not isinstance(content, str):
        return 0
    scores = analyzer.polarity_scores(content)
    compound_score = scores['compound']
    
    if abs(compound_score) >= VADER_THRESHOLD:
        return 1
    
    return 0 #0 for unbiased and 1 for biased
df['bias_label_vader'] = df['content'].apply(relabel_with_vader)

print(f"New Target Class Counts (df['bias_label_vader']):\n{df['bias_label_vader'].value_counts()}")

New Target Class Counts (df['bias_label_vader']):
bias_label_vader
1    6245
0     959
Name: count, dtype: int64


In [210]:
df

,date,year,month,day,tag,loc,headline,title,content,polarity,subjectivity,bias_label,bias_label_vader
0,20181010,2018,1,10,business,Mumbai,india business_bob partners truecaller for upi...,20181010_business_india-business_bob-partners-...,Mumbai: Bank of Baroda (BoB) has tied-up with ...,0.293290,0.451491,0,1
1,20181010,2018,1,10,business,,india business_city to get a centre of excelle...,20181010_business_india-business_city-to-get-a...,Chennai will soon be home to a Centre of Excel...,-0.026154,0.355000,0,1
2,20181010,2018,1,10,business,,india business_cos should offer flexible work ...,20181010_business_india-business_cos-should-of...,"By Kamal Karanth According to Aristotle, ‘The ...",0.213180,0.475056,0,1
3,20181010,2018,1,10,business,CHENNAI,india business_deal flows currency move to hel...,20181010_business_india-business_deal-flows-cu...,CHENNAI: The traditionally strong quarter (Jul...,0.102381,0.409524,0,1
4,20181010,2018,1,10,business,Coimbatore,india business_equity mutual funds remain bull...,20181010_business_india-business_equity-mutual...,Coimbatore: FIIs (foreign institutional invest...,0.133001,0.397082,0,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...
7199,20181027,2018,1,27,city_pune,PUNE,alert residents nab duo for robbing carpenter.,20181027_city_pune_alert-residents-nab-duo-for...,PUNE: Alert residents nabbed a man and his min...,0.021726,0.346726,0,1
7200,20181027,2018,1,27,city_pune,PUNE,as festival nears prices of bajra wheat jowar ...,20181027_city_pune_as-festival-nears-prices-of...,"PUNE: After onions, it’s now the turn of wheat...",0.024463,0.341648,0,1
7201,20181027,2018,1,27,city_pune,PUNE,eco friendly twist to this years lanterns for ...,20181027_city_pune_eco-friendly-twist-to-this-...,PUNE: Shops lined in the Raviwar Peth market a...,0.153783,0.447407,0,0
7202,20181027,2018,1,27,city_pune,PUNE,ferreira gonsalves back in police custody afte...,20181027_city_pune_ferreira-gonsalves-back-in-...,PUNE: The city police on Friday evening took a...,-0.005102,0.302551,0,1


**PIPELINE TRAINING**

In [211]:
X = df[['content']].copy()
y = df['bias_label_vader']

In [212]:
valid_mask = X['content'].notna() & (X['content'].apply(lambda x: isinstance(x, str))) & (X['content'].str.len() > 0)

In [213]:
X_cleaned = X[valid_mask].copy()
y_cleaned = y[valid_mask].copy()

In [214]:
X_train, X_test, y_train, y_test = train_test_split(
    X_cleaned, y_cleaned, test_size=0.2, random_state=42, stratify=y_cleaned
)

In [215]:
print("X_train type:", type(X_train))
print("X_train columns:", X_train.columns.tolist())

X_train type: <class 'pandas.core.frame.DataFrame'>
X_train columns: ['content']


In [216]:
from custom_utils import VaderSentimentExtractor 
from custom_utils import sparse_to_dense

In [217]:
df

,date,year,month,day,tag,loc,headline,title,content,polarity,subjectivity,bias_label,bias_label_vader
0,20181010,2018,1,10,business,Mumbai,india business_bob partners truecaller for upi...,20181010_business_india-business_bob-partners-...,Mumbai: Bank of Baroda (BoB) has tied-up with ...,0.293290,0.451491,0,1
1,20181010,2018,1,10,business,,india business_city to get a centre of excelle...,20181010_business_india-business_city-to-get-a...,Chennai will soon be home to a Centre of Excel...,-0.026154,0.355000,0,1
2,20181010,2018,1,10,business,,india business_cos should offer flexible work ...,20181010_business_india-business_cos-should-of...,"By Kamal Karanth According to Aristotle, ‘The ...",0.213180,0.475056,0,1
3,20181010,2018,1,10,business,CHENNAI,india business_deal flows currency move to hel...,20181010_business_india-business_deal-flows-cu...,CHENNAI: The traditionally strong quarter (Jul...,0.102381,0.409524,0,1
4,20181010,2018,1,10,business,Coimbatore,india business_equity mutual funds remain bull...,20181010_business_india-business_equity-mutual...,Coimbatore: FIIs (foreign institutional invest...,0.133001,0.397082,0,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...
7199,20181027,2018,1,27,city_pune,PUNE,alert residents nab duo for robbing carpenter.,20181027_city_pune_alert-residents-nab-duo-for...,PUNE: Alert residents nabbed a man and his min...,0.021726,0.346726,0,1
7200,20181027,2018,1,27,city_pune,PUNE,as festival nears prices of bajra wheat jowar ...,20181027_city_pune_as-festival-nears-prices-of...,"PUNE: After onions, it’s now the turn of wheat...",0.024463,0.341648,0,1
7201,20181027,2018,1,27,city_pune,PUNE,eco friendly twist to this years lanterns for ...,20181027_city_pune_eco-friendly-twist-to-this-...,PUNE: Shops lined in the Raviwar Peth market a...,0.153783,0.447407,0,0
7202,20181027,2018,1,27,city_pune,PUNE,ferreira gonsalves back in police custody afte...,20181027_city_pune_ferreira-gonsalves-back-in-...,PUNE: The city police on Friday evening took a...,-0.005102,0.302551,0,1


In [252]:
to_dense_transformer = FunctionTransformer(
    sparse_to_dense,
    accept_sparse=True 
)

In [253]:
to_dense_transformer

FunctionTransformer(accept_sparse=True,
                    func=<function sparse_to_dense at 0x0000017D3590D4E0>)

In [254]:
preprocessor = ColumnTransformer(
    transformers=[
        ('text_pipeline', TfidfVectorizer(max_features=4000, stop_words='english', ngram_range=(1, 2)), 'content'),
        ('vader_pipeline', Pipeline([
            ('vader_extractor', VaderSentimentExtractor()),
            ('scaler', StandardScaler())
        ]), 'content')
    ],
    remainder='drop'
)


In [255]:
preprocessor

ColumnTransformer(transformers=[('text_pipeline',
                                 TfidfVectorizer(max_features=4000,
                                                 ngram_range=(1, 2),
                                                 stop_words='english'),
                                 'content'),
                                ('vader_pipeline',
                                 Pipeline(steps=[('vader_extractor',
                                                  VaderSentimentExtractor()),
                                                 ('scaler', StandardScaler())]),
                                 'content')])

In [256]:
from sklearn.svm import SVC
from sklearn.pipeline import Pipeline

pipeline_svc = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('to_dense', to_dense_transformer),
    ('classifier', SVC(kernel='rbf', probability=True, random_state=42))
])

pipeline_svc.fit(X_train, y_train)
y_pred_svc = pipeline_svc.predict(X_test)
accuracy_score(y_test,y_pred_svc)

pipeline_svc.fit(X_train, y_train)

y_pred_train_svc = pipeline_svc.predict(X_train)
y_pred_test_svc = pipeline_svc.predict(X_test)

train_accuracy_svc = accuracy_score(y_train, y_pred_train_svc)
test_accuracy_svc = accuracy_score(y_test, y_pred_test_svc)

print(f"Training Accuracy: {train_accuracy_svc:.4f}")
print(f"Test Accuracy:     {test_accuracy_svc:.4f}")

Training Accuracy: 0.9993
Test Accuracy:     0.9868


In [268]:
accuracy_score(y_test,y_pred_svc)

0.9868147120055517

In [257]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.pipeline import Pipeline

pipeline_dt = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('to_dense', to_dense_transformer),
    ('classifier', DecisionTreeClassifier(random_state=42))
])

pipeline_dt.fit(X_train, y_train)
y_pred_dt = pipeline_dt.predict(X_test)
accuracy_score(y_test,y_pred_dt)

pipeline_dt.fit(X_train, y_train)

y_pred_train_dt = pipeline_dt.predict(X_train)
y_pred_test_dt = pipeline_dt.predict(X_test)

train_accuracy_dt = accuracy_score(y_train, y_pred_train_dt)
test_accuracy_dt = accuracy_score(y_test, y_pred_test_dt)

print(f"Training Accuracy: {train_accuracy_dt:.4f}")
print(f"Test Accuracy:     {test_accuracy_dt:.4f}")

Training Accuracy: 1.0000
Test Accuracy:     1.0000


In [269]:
accuracy_score(y_test,y_pred_dt)

1.0

In [258]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline

pipeline_rf = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('to_dense', to_dense_transformer),
    ('classifier', RandomForestClassifier(random_state=42, n_estimators=100))
])

pipeline_rf.fit(X_train, y_train)
y_pred_rf = pipeline_rf.predict(X_test)
accuracy_score(y_test,y_pred_rf)

pipeline_rf.fit(X_train, y_train)

y_pred_train_rf = pipeline_rf.predict(X_train)
y_pred_test_rf = pipeline_rf.predict(X_test)

train_accuracy_rf = accuracy_score(y_train, y_pred_train_rf)
test_accuracy_rf = accuracy_score(y_test, y_pred_test_rf)

print(f"Training Accuracy: {train_accuracy_rf:.4f}")
print(f"Test Accuracy:     {test_accuracy_rf:.4f}")

Training Accuracy: 1.0000
Test Accuracy:     0.9542


In [270]:
accuracy_score(y_test,y_pred_rf)

0.9541984732824428

In [259]:
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline

pipeline_logreg = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('to_dense', to_dense_transformer),
    ('classifier', LogisticRegression(random_state=42, solver='liblinear'))
])

pipeline_logreg.fit(X_train, y_train)
y_pred_logreg = pipeline_logreg.predict(X_test)
accuracy_score(y_test,y_pred_logreg)

pipeline_logreg.fit(X_train, y_train)

y_pred_train_logreg = pipeline_logreg.predict(X_train)
y_pred_test_logreg = pipeline_logreg.predict(X_test)

train_accuracy_logreg = accuracy_score(y_train, y_pred_train_logreg)
test_accuracy_logreg = accuracy_score(y_test, y_pred_test_logreg)

print(f"Training Accuracy: {train_accuracy_logreg:.4f}")
print(f"Test Accuracy:     {test_accuracy_logreg:.4f}")

Training Accuracy: 0.8804
Test Accuracy:     0.8688


In [271]:
accuracy_score(y_test,y_pred_logreg)

0.8688410825815406

In [260]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline

pipeline_knn = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('to_dense', to_dense_transformer),
    ('classifier', KNeighborsClassifier(n_neighbors=5))
])

pipeline_knn.fit(X_train, y_train)
y_pred_knn = pipeline_knn.predict(X_test)
accuracy_score(y_test,y_pred_knn)

pipeline_knn.fit(X_train, y_train)

y_pred_train_knn = pipeline_knn.predict(X_train)
y_pred_test_knn = pipeline_knn.predict(X_test)

train_accuracy_knn = accuracy_score(y_train, y_pred_train_knn)
test_accuracy_knn = accuracy_score(y_test, y_pred_test_knn)

print(f"Training Accuracy: {train_accuracy_knn:.4f}")
print(f"Test Accuracy:     {test_accuracy_knn:.4f}")

Training Accuracy: 0.9818
Test Accuracy:     0.9681


In [272]:
accuracy_score(y_test,y_pred_knn)

0.9680777238029147

**AFTER TRAINING 5 DIFFERENT MODELS WE FOUND OUT DECISION TREE WITH HIGHEST ACCURACY**

**THEREFORE WE USED DECISION TREE TO PREDICT UNSEEN DATA (NEWS ARTICLES)**

In [263]:
new_articles_set = [
    # Article A: Factual Report (Expected: Unbiased / Low Probability)
    """The company announced a 3% decrease in profit for the third quarter, 
    aligning with the revised expectations provided by management last month.""",
    
    # Article B: Emotional Critique (Expected: Biased / High Probability)
    """The company's catastrophic 3% plunge in profits is a monumental failure 
    that reveals utter incompetence at the executive level and warrants immediate 
    termination of the entire leadership team!""",
    
    # Article C: Standard Business Update (Expected: Unbiased / Low Probability)
    """Oil futures closed slightly lower today amidst cautious trading 
    ahead of a major OPEC meeting scheduled for next week."""

    "Global Tech Inc. today reported its third-quarter earnings, announcing a revenue of $4.5 billion, a 5% increase from the same period last year. Net income stood at $850 million. The company cited strong sales in its software division as the primary driver of growth. The earnings per share met the consensus estimate from market analysts."

    "The Ministry of Transportation confirmed that construction on the new 25-kilometer coastal highway will begin on November 1st. The project, valued at $300 million, is expected to take two years to complete and aims to reduce traffic congestion between the two major cities."

    "The administration's latest economic policy is a catastrophic failure that demonstrates a shocking level of incompetence. This reckless decision will inevitably crush small businesses under a mountain of absurd regulations, proving once again that the leadership is completely out of touch with the struggles of ordinary people."

    "Innovate Corp has just released its groundbreaking new smartphone, and it is nothing short of a revolutionary masterpiece. The device's flawless design and genius-level software deliver a user experience so perfect it will change the world. This isn't just a phone; it's a monumental leap forward for all of humanity."

    "Brave activists heroically gathered today to demand long-overdue justice from stubborn city officials. Despite the passionate and heartfelt pleas from the community, the cold-hearted bureaucracy refused to even listen, shamelessly dismissing the valid concerns of the very people they are supposed to serve."
]

In [264]:
predictions = predict_bias_pipeline(new_articles_set, pipeline_dt)

Article 1: 0.0000
Article 2: 1.0000
Article 3: 1.0000


In [265]:
print("\nFinal Predicted Labels:", predictions)


Final Predicted Labels: [0 1 1]


In [266]:
labels = {0: 'Unbiased (0)', 1: 'Biased (1)'}

In [267]:
Predicted_Label = [labels[p] for p in optimized_predictions]
Predicted_Label

['Unbiased (0)', 'Biased (1)', 'Unbiased (0)']

In [184]:
import joblib
filename = 'media_bias_prediction_model.joblib'

In [185]:
joblib.dump(pipeline, filename)

['media_bias_prediction_model.joblib']